In [ ]:
from supabase import create_client
import os

# Điền trực tiếp hoặc lấy từ env
SUPABASE_URL = "https://cvdoasxazyruytejluvv.supabase.co"
SUPABASE_KEY = ""

client = create_client(SUPABASE_URL, SUPABASE_KEY)

In [ ]:
def fetch_all(table, select_cols, page_size=1000, query_builder=None):
    all_rows = []
    page = 0
    while True:
        q = client.table(table).select(select_cols)
        if query_builder:
            q = query_builder(q)
        resp = q.range(page * page_size, (page + 1) * page_size - 1).execute()
        rows = resp.data or []
        if not rows:
            break
        all_rows.extend(rows)
        if len(rows) < page_size:
            break
        page += 1
    return all_rows

approved_images = fetch_all(
    "image",
    "image_id, image_desc, food_items, is_checked, is_drop",
    query_builder=lambda q: q.eq("is_checked", True).eq("is_drop", False)
)

vqa_rows = fetch_all("vqa", "image_id")

approved_ids = {r["image_id"] for r in approved_images if r.get("image_id")}
vqa_ids = {r["image_id"] for r in vqa_rows if r.get("image_id")}
missing_ids = sorted(approved_ids - vqa_ids)

print("Approved images:", len(approved_ids))
print("Images appearing in vqa:", len(vqa_ids))
print("Missing approved images:", len(missing_ids))
print("First 20 missing IDs:", missing_ids[:20])

# phân loại nhanh theo điều kiện fetch của script
missing_rows = [r for r in approved_images if r.get("image_id") in set(missing_ids)]

missing_desc = [r["image_id"] for r in missing_rows if not r.get("image_desc")]
missing_items = [r["image_id"] for r in missing_rows if not r.get("food_items")]

print("\nAmong missing approved images:")
print(" - missing image_desc:", len(missing_desc))
print(" - missing food_items:", len(missing_items))

In [ ]:
from pathlib import Path
import json

out_dir = Path("data/vqa_rerun_missing")
out_dir.mkdir(parents=True, exist_ok=True)

txt_path = out_dir / "missing_image_ids.txt"
json_path = out_dir / "missing_image_ids.json"

txt_path.write_text("\n".join(missing_ids), encoding="utf-8")
json_path.write_text(json.dumps(missing_ids, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved:", txt_path)
print("Saved:", json_path)
print("Count:", len(missing_ids))